# Nonlinear convex transaction costs

This notebook replaces the quadratic-only trading model with a separable power-law impact cost

$$
c(t)=\sum_i\eta_i\left[(t_i^2+\epsilon^2)^{p/2}-\epsilon^p\right],
\qquad t=h-h_-.
$$

For $p=3/2$ this is a smooth approximation to square-root-impact total cost. The smoothing parameter
$\epsilon>0$ avoids the divergent curvature of $|t|^{3/2}$ at zero without changing convexity.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.float_format", lambda value: f"{value:,.8g}")

In [ ]:
from portfolio_pgd import (
    ConstraintSet,
    PGDOptions,
    PortfolioProblem,
    PowerLawCost,
    capped_long_only_portfolio,
    factor_covariance,
    solve_pgd,
    solve_scipy_slsqp,
)

n_assets = 30
rng = np.random.default_rng(2201)
covariance, _ = factor_covariance(n_assets, 4, seed=2202, specific_risk=0.15)
previous = capped_long_only_portfolio(n_assets, cap=0.055, seed=2203)
eta = 0.008 + 0.012 * rng.random(n_assets)

cost = PowerLawCost(eta=eta, p=1.5, epsilon=1e-3)
problem = PortfolioProblem(
    alpha=rng.normal(scale=0.03, size=n_assets),
    covariance=covariance,
    previous_holdings=previous,
    risk_aversion=1.8,
    quadratic_cost_matrix=0.2 + rng.random(n_assets),
    quadratic_cost_aversion=0.25,
    nonlinear_cost=cost,
)
constraints = ConstraintSet(
    n_assets,
    equality_matrix=np.ones((1, n_assets)),
    equality_target=np.array([1.0]),
    lower_bounds=0.0,
    upper_bounds=0.075,
)

pgd = solve_pgd(
    problem,
    constraints,
    options=PGDOptions(max_iterations=25_000, tolerance=5e-8),
)
slsqp = solve_scipy_slsqp(problem, constraints)
print(f"PGD: {pgd.status} in {pgd.iterations} iterations")
print(f"SLSQP: success={slsqp.success}; {slsqp.message}")

## Independent solver comparison

The nonlinear objective is convex but no longer quadratic. Therefore the notebook compares PGD to
SciPy SLSQP rather than to a linear KKT solve.

In [ ]:
comparison = pd.DataFrame(
    {
        "objective": [pgd.objective, slsqp.objective],
        "utility": [pgd.utility, -slsqp.objective],
        "turnover": [np.sum(np.abs(pgd.trades)), np.sum(np.abs(slsqp.holdings - previous))],
        "distance_to_SLSQP": [np.linalg.norm(pgd.holdings - slsqp.holdings), 0.0],
        "constraint_violation": [
            constraints.max_violation(pgd.holdings),
            constraints.max_violation(slsqp.holdings),
        ],
    },
    index=["PGD", "SciPy SLSQP"],
)
print(comparison.to_string())

## Cost, marginal cost, and curvature

The gradient enters PGD directly. The Hessian diagonal is used only to initialize a conservative
step; the majorization line search supplies the actual global safeguard.

In [ ]:
trade_grid = np.linspace(-0.08, 0.08, 401)
unit_cost = PowerLawCost(eta=1.0, p=1.5, epsilon=1e-3)
cost_values = np.array([unit_cost.value(np.array([trade])) for trade in trade_grid])
marginal = np.array([unit_cost.gradient(np.array([trade]))[0] for trade in trade_grid])
curvature = np.array([unit_cost.hessian_diag(np.array([trade]))[0] for trade in trade_grid])

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(trade_grid, cost_values)
axes[0].set(title="Power-law cost", xlabel="Trade", ylabel="Cost / eta")
axes[1].plot(trade_grid, marginal)
axes[1].set(title="Marginal cost", xlabel="Trade", ylabel="dc/dt / eta")
axes[2].plot(trade_grid, curvature)
axes[2].set(title="Local curvature", xlabel="Trade", ylabel="d²c/dt² / eta")
plt.tight_layout()
plt.show()

## Convergence and the realized trade distribution

In [ ]:
history = pd.DataFrame(pgd.history)
valid = history["projected_gradient_norm"].notna()
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].semilogy(
    history.loc[valid, "iteration"],
    np.maximum(history.loc[valid, "projected_gradient_norm"], 1e-18),
)
axes[0].set(title="Projected-gradient residual", xlabel="Iteration", ylabel="Norm")
axes[1].bar(np.arange(n_assets), pgd.trades)
axes[1].axhline(0.0, color="black", linewidth=0.8)
axes[1].set(title="Optimal trades", xlabel="Asset", ylabel="Weight change")
plt.tight_layout()
plt.show()

## Sensitivity to the impact exponent

Holding every other input fixed, we resolve the portfolio for several convex exponents. This is an
algorithmic comparison—not a claim that any one exponent is universally appropriate.

In [ ]:
rows = []
for exponent in [1.25, 1.5, 2.0, 3.0]:
    exponent_problem = PortfolioProblem(
        alpha=problem.alpha,
        covariance=problem.covariance,
        previous_holdings=previous,
        risk_aversion=problem.risk_aversion,
        quadratic_cost_matrix=problem.quadratic_cost_matrix,
        quadratic_cost_aversion=problem.quadratic_cost_aversion,
        nonlinear_cost=PowerLawCost(eta=eta, p=exponent, epsilon=1e-3),
    )
    solved = solve_pgd(
        exponent_problem,
        constraints,
        options=PGDOptions(max_iterations=25_000, tolerance=2e-7),
    )
    rows.append(
        {
            "p": exponent,
            "converged": solved.converged,
            "utility": solved.utility,
            "turnover": np.sum(np.abs(solved.trades)),
            "max_abs_trade": np.max(np.abs(solved.trades)),
            "iterations": solved.iterations,
        }
    )
sensitivity = pd.DataFrame(rows).set_index("p")
print(sensitivity.to_string())

## Executable acceptance tests

In [ ]:
assert pgd.converged
assert slsqp.success
assert constraints.max_violation(pgd.holdings) < 2e-8
assert abs(pgd.objective - slsqp.objective) < 2e-7
assert np.linalg.norm(pgd.holdings - slsqp.holdings) < 8e-4
assert sensitivity["converged"].all()
print("All nonlinear-cost validation checks passed.")